创建实体微调数据集

In [ ]:
import json
import torch
import random
from sklearn.model_selection import train_test_split  # 用于划分数据集

# 加载数据集
dataset = torch.load('qm9_dataset_small.pt')

# 准备微调数据
finetune_data = []

prompt='!!Rules must be followed: 1. Extract the molecular formula (in SMILES format) from the given sentence. 2. Just generate the answer, dont show the process or method. 3. The molecular formula can be represented in SMILES notation. 4. Your task is to identify and extract the molecular formula (SMILES) from the provided question.'

inputs = [
    "Generate the position tensor `pos` and atomic number tensor `z` for the molecule with the chemical formula {smile}.",
    "Create the position tensor `pos` and the atomic number tensor `z` for the molecule represented by the chemical formula {smile}.",
    "Produce the position tensor `pos` and atomic number tensor `z` corresponding to the molecule with the chemical formula {smile}.",
    "Calculate the position tensor `pos` and atomic number tensor `z` for the molecule with the chemical formula {smile}.",
    "Generate the `pos` and `z` tensors for the molecule with the chemical structure {smile}.",
    "Provide the position tensor `pos` and atomic number tensor `z` for the molecule defined by the chemical formula {smile}.",
    "Generate the molecular position tensor `pos` and atomic number tensor `z` for the molecule represented by {smile}."
]


# 遍历数据集并生成条目
for data in dataset:
    smile = data['smile']  # 提取smile值
    input = random.choice(inputs)  # 随机选择一个指令

    # 将 pos 和 z 张量中的浮点数转换为科学计数法并保留五位有效数字
    pos = [[f"{coord:.4e}" for coord in atom] for atom in data['pos'].tolist()]
    z = data['z'].tolist()

    entry = {
        "Instruction": prompt,  # 格式化指令
        "Input": input.format(smile=smile),  # 输入为空，或者可以根据需要填充
        "Output": f"{smile}"  # 格式化为字符串，并加入pos和z张量
    }
    finetune_data.append(entry)

# 将数据分为训练集、验证集和测试集（60%训练集，20%验证集，20%测试集）
train_data, valid_data = train_test_split(finetune_data, test_size=0.2, random_state=42)  # 60%用于训练

# 保存为JSON文件
with open('Entity_train.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

with open('Entity_valid.json', 'w', encoding='utf-8') as f:
    json.dump(valid_data, f, ensure_ascii=False, indent=4)

print("数据集拆分完成：60%训练集，20%验证集，20%测试集。")


实体化学式微调

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
from datasets import load_dataset
import matplotlib.pyplot as plt
import logging

# 配置日志记录
logging.basicConfig(
    filename='training_log.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def preprocess_logits_for_metrics(logits, labels):
    """
    Original Trainer may have a memory leak. 
    This is a workaround to avoid storing too many tensors that are not needed.
    """
    pred_ids = torch.argmax(logits, dim=-1)
    return pred_ids, labels

# 配置模型和数据集路径
MODEL_NAME='Qwen/Qwen2.5-Math-1.5B-Instruct'
TRAIN_DATASET_PATH = "Finetune_data\Entity_train.json"
EVAL_DATASET_PATH = "Finetune_data\Entity_valid.json"

OUTPUT_DIR='Finetune_LLM\qwen2.5_math_Ent'

# 加载模型和分词器
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side='right')
model_ori = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)

# 配置 LoRA 微调参数
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    r=8,
    lora_alpha=16,
    lora_dropout=0.01,
)

# 应用 LoRA 配置到模型
model = get_peft_model(model_ori, peft_config)

# 数据集加载
train_dataset = load_dataset('json', data_files=TRAIN_DATASET_PATH)['train']
eval_dataset = load_dataset('json', data_files=EVAL_DATASET_PATH)['train']

# Tokenization
def process_func(batch):
    MAX_LENGTH = 256
    input_ids, attention_masks, labels = [], [], []
    
    for i in range(len(batch['Instruction'])):
        instruction = tokenizer(
            f"<|im_start|>system\nYou are a scientist to extract the smile formula of the organic molecule mentioned in the following words.<|im_end|>\n<|im_start|>user\n{batch['Instruction'][i] + batch['Input'][i]}<|im_end|>\n<|im_start|>assistant\n",
            add_special_tokens=False
        )
        response = tokenizer(f"{batch['Output'][i]}", add_special_tokens=False)
        input_id = instruction["input_ids"] + response["input_ids"] + [tokenizer.pad_token_id]
        attention_mask = instruction["attention_mask"] + response["attention_mask"] + [1]
        label = [-100] * len(instruction["input_ids"]) + response["input_ids"] + [tokenizer.pad_token_id]

        # 截断或填充到最大长度
        if len(input_id) > MAX_LENGTH:
            input_id = input_id[:MAX_LENGTH]
            attention_mask = attention_mask[:MAX_LENGTH]
            label = label[:MAX_LENGTH]
        else:
            padding_length = MAX_LENGTH - len(input_id)
            input_id += [tokenizer.pad_token_id] * padding_length
            attention_mask += [0] * padding_length
            label += [-100] * padding_length

        input_ids.append(input_id)
        attention_masks.append(attention_mask)
        labels.append(label)

    # 返回处理后的数据
    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels
    }

# Tokenize 数据集
tokenized_train = train_dataset.map(process_func, batched=True)
tokenized_eval = eval_dataset.map(process_func, batched=True)

# 设置训练参数
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    logging_steps=200,
    evaluation_strategy="steps",
    eval_steps=100,
    save_steps=500,
    logging_dir="./logs",
    remove_unused_columns=False,
)

# 自定义 Trainer 用于记录训练和验证集的损失
class CustomTrainer(Trainer):
    def log(self, logs: dict, *args, **kwargs) -> None:
        super().log(logs, *args, **kwargs)  # 调用父类的 log 方法
        if 'loss' in logs:
            step = self.state.global_step
            train_loss = logs['loss']
            logging.info(f"Step: {step}, Train Loss: {train_loss}")
            if step % training_args.eval_steps == 0:
                 eval_results = self.evaluate()
                 eval_loss = eval_results.get('eval_loss', "N/A")
                 logging.info(f"Step: {step}, Eval Loss: {eval_loss}")


# 训练模型
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

trainer.train()

实体微调推理测试


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
from bert_score import score
from rouge_score import rouge_scorer
import json
import logging

# 配置日志记录
logging.basicConfig(
    filename='inference_log.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# 配置模型和数据集路径
MODEL_PATH = 'Qwen/Qwen2.5-Math-1.5B-Instruct'
TEST_DATASET_PATH = "Finetune_data\Entity_valid.json"
LORA_PATH = 'Finetune_LLM\qwen2.5_math_Ent\checkpoint-8000'

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True)

# 加载LoRA微调权重
model = PeftModel.from_pretrained(model, model_id=LORA_PATH)

# 将模型和数据传送到NPU设备
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# 数据集加载
test_dataset = load_dataset('json', data_files=TEST_DATASET_PATH)['train']

# 推理函数：基于模型生成输出
def generate_inference(input_data):
    prompt = input_data['Instruction'] + input_data['Input']
    messages = [
        {"role": "system", "content": "Extract the organic molecule mentioned in the following words."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize and transfer to device
    model_inputs = tokenizer([text], return_tensors="pt", padding=True, truncation=True)

    # Manually create attention mask if necessary
    input_ids = model_inputs.input_ids
    attention_mask = model_inputs.attention_mask

    # Transfer tensors to the device (GPU/CPU)
    model_inputs = {key: value.to(device) for key, value in model_inputs.items()}

    # 生成预测输出
    generated_ids = model.generate(
        model_inputs['input_ids'],
        attention_mask=model_inputs['attention_mask'],  # Pass attention_mask explicitly
        max_new_tokens=120
    )
    
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs['input_ids'], generated_ids)
    ]

    # 解码生成的文本
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

# 收集生成的文本和实际的输出文本
generated_outputs = []
actual_outputs = []
input_data_list = []

for example in test_dataset:
    generated_text = generate_inference(example)
    generated_outputs.append(generated_text)
    print('generate:',generated_text)
    actual_outputs.append(example['Output'])
    input_data_list.append({
        'Instruction': example['Instruction'],
        'Input': example['Input'],
        'Generated Output': generated_text,
        'Actual Output': example['Output']
    })

# 保存推理输入和输出到文件
with open('generate_output.json', 'w') as f:
    json.dump(input_data_list, f, indent=4)

# 计算 BERTScore
P, R, F1 = score(generated_outputs, actual_outputs, lang='en', verbose=True)

# 输出 BERTScore 结果
logging.info(f"----- BERTScore -----")
logging.info(f"Precision: {P.mean().item():.4f}")
logging.info(f"Recall: {R.mean().item():.4f}")
logging.info(f"F1 Score: {F1.mean().item():.4f}")

# 计算 ROUGE 分数
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge_scores = []
for generated, actual in zip(generated_outputs, actual_outputs):
    score = scorer.score(actual, generated)
    rouge_scores.append(score)

rouge1 = sum([score['rouge1'].fmeasure for score in rouge_scores]) / len(rouge_scores)
rouge2 = sum([score['rouge2'].fmeasure for score in rouge_scores]) / len(rouge_scores)
rougeL = sum([score['rougeL'].fmeasure for score in rouge_scores]) / len(rouge_scores)

# 输出 ROUGE 分数
logging.info(f"----- ROUGE Scores -----")
logging.info(f"ROUGE-1: {rouge1:.4f}")
logging.info(f"ROUGE-2: {rouge2:.4f}")
logging.info(f"ROUGE-L: {rougeL:.4f}")

# 打印评估结果到控制台
print("----- BERTScore -----")
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall: {R.mean().item():.4f}")
print(f"F1 Score: {F1.mean().item():.4f}")

print("----- ROUGE Scores -----")
print(f"ROUGE-1: {rouge1:.4f}")
print(f"ROUGE-2: {rouge2:.4f}")
print(f"ROUGE-L: {rougeL:.4f}")
